# 13 - Splink Demonstration (train -> predict -> cluster -> evaluate)

Notebook end-to-end yang memakai adapter `src/splink_pipeline.py` dan CLI
`src/splink_cli.py` untuk menjalankan dedupe probabilistic dengan Splink
menggunakan label manual dari `manual_review_queue.csv` (107 pair: 25 match, 82 non-match).

Alur: ku demi bridge label -> train m/u -> predict pairwise -> tune threshold -> cluster -> evaluate.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from importlib.util import find_spec

assert find_spec('splink'), 'pip install -e ".[splink]"'

from src.splink_pipeline import (
    load_reviewed_labels,
    train_splink_pipeline,
    cluster_predictions,
    evaluate_splink_predictions,
    tune_splink_threshold,
    prepare_splink_input,
)
from src.io import load_customer_csv

INPUT = Path('data/raw/crm_50000_customers_dirty_v3.csv')
QUEUE = Path('data/processed/manual_review_queue.csv')
OUT_DIR = Path('data/processed/splink_demo')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('input:', INPUT.exists(), '| queue:', QUEUE.exists())

## 1. Bridge label manual -> format Splink

`manual_review_queue.csv` memakai `left_row_index/right_row_index/review_label`.
`load_reviewed_labels` membaca langsung dan menormalkan ke `record_id_l/record_id_r/clerical_match_score`
tanpa memakai `customer_id` sebagai ground truth.

In [ ]:
labels = load_reviewed_labels(QUEUE)
print('label shape:', labels.shape)
print('distribution:\n', labels['clerical_match_score'].value_counts())
print(labels.head())

## 2. Train m/u dan predict pairwise

Model di-train dari label. WARNING tentang sebagian m/u `name_key_std` dan `address_std`
dapat muncul karena sample label didominasi pasangan yang ber-agree penuh; ini wajar.

In [ ]:
predictions = train_splink_pipeline(INPUT, QUEUE, OUT_DIR / 'splink_predictions.csv')
print('predictions:', len(predictions), 'pairs')
print(predictions['match_probability'].describe())

## 3. Tuning threshold + evaluasi di reviewed labels

Metrik hanya dihitung di sample label, bukan seluruh candidate set (tidak ada ground truth global).

In [ ]:
best_threshold, eval_summary = tune_splink_threshold(
    predictions, labels, thresholds=[0.1, 0.3, 0.5, 0.7, 0.9]
)
eval_summary = eval_summary.sort_values('threshold')
print('best threshold by F1:', best_threshold)
display(eval_summary[['threshold', 'true_positive', 'false_positive',
                       'false_negative', 'precision', 'recall', 'f1']])

In [ ]:
ax = eval_summary.plot.line(x='threshold', y=['precision', 'recall', 'f1'], marker='o')
ax.set_title('Splink metrics vs threshold (reviewed labels only)')
ax.axvline(best_threshold, color='gray', ls='--')
ax.figure.tight_layout()

## 4. Cluster pairwise -> entity

Gunakan connected components di threshold terbaik. Setiap `record_id` diberi `cluster_id`;
record yang merepresentasikan entity sama menyatu dalam satu cluster.

In [ ]:
standardized = prepare_splink_input(load_customer_csv(INPUT))
clusters = cluster_predictions(standardized, predictions, threshold=best_threshold)
clusters.to_csv(OUT_DIR / 'splink_clusters.csv', index=False)

print('records:', len(clusters), '| clusters:', clusters['cluster_id'].nunique())
print('duplicated records (percent):')
dedupe_count = len(clusters) - clusters['cluster_id'].nunique()
print(f'  {dedupe_count} records ({dedupe_count/len(clusters):.2%}) tergabung dalam cluster multi-record')
clusters['cluster_id'].value_counts().value_counts().sort_index().rename_axis('records_per_cluster').reset_index(name='n_clusters')

## 5. Artifact
Semua hasil tersimpan di `data/processed/splink_demo/`:

In [ ]:
for path in sorted(OUT_DIR.glob('splink_*')):
    print('-' , path.name, path.stat().st_size, 'bytes')